In [3]:
import sys
from pathlib import Path

def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()



[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [9]:
from pathlib import Path
import json

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


In [ ]:
# ---------------------------------------------------------
# Load precomputed train/test sequences for the LSTM
# ---------------------------------------------------------
# Earlier in the pipeline we generated:
#   - X_train.npy : training input sequences
#   - y_train.npy : training labels
#   - X_test.npy  : test input sequences
#   - y_test.npy  : test labels
#   - meta.json   : metadata (feature names, sequence length, etc.)
#
# We now simply load these NumPy arrays back into memory so the LSTM
# training script can use them directly.
#
# NOTE:
#   • Using .npy keeps array shapes and dtypes intact.
#   • This avoids recomputing sequences every time we train.
#   • The directory 03_Sequences acts as a reproducible “data build” stage.
# ---------------------------------------------------------

# Base project root (train_lstm.py is inside 04_Models/, so one level up)
ROOT = Path.cwd().parent             # 04_train -> project root

# Directory containing the saved sequences and metadata
SEQ_DIR = ROOT / "03_Sequences"

# Paths to the saved NumPy arrays and metadata file
X_train_path = SEQ_DIR / "X_train.npy"
y_train_path = SEQ_DIR / "y_train.npy"
X_test_path  = SEQ_DIR / "X_test.npy"
y_test_path  = SEQ_DIR / "y_test.npy"
meta_path    = SEQ_DIR / "meta.json"

print("Using:")
print("  ", X_train_path)
print("  ", y_train_path)
print("  ", X_test_path)
print("  ", y_test_path)
print("  ", meta_path)

# ---------------------------------------------------------
# Load the datasets into memory
# ---------------------------------------------------------
# These arrays have the shapes:
#   X_train : (num_train_samples, seq_len, num_features)
#   y_train : (num_train_samples,)
#   X_test  : (num_test_samples,  seq_len, num_features)
#   y_test  : (num_test_samples,)
#
# They contain exactly the same data produced previously in the
# feature-building + sequence-building notebook.
# ---------------------------------------------------------
X_train = np.load(X_train_path)
y_train = np.load(y_train_path)
X_test  = np.load(X_test_path)
y_test  = np.load(y_test_path)

# ---------------------------------------------------------
# Load metadata including:
#   - feature_cols (list of input features)
#   - seq_len      (window length, e.g. 60)
#   - scaler params (if you saved them)
#   - timestamps or any other contextual info
#
# This ensures the training script “knows” how the sequences were built.
# ---------------------------------------------------------
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

# Show shapes to verify everything is loaded correctly
print("meta:", meta)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


Using:
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/X_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/y_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/X_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/y_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/meta.json
meta: {'seq_len': 60, 'feature_cols': ['log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm', 'volume_ratio_5m', 'volume_ratio_15m', 'log_re

In [11]:
# -------------------------------------------------------------
# Create a validation set from the *end* of the training set
# -------------------------------------------------------------
# We already performed a strict time-based split:
#   • Training data = earlier part of the dataset
#   • Test data     = later, unseen part
#
# Now we take ~10% of the *training* portion and reserve it as
# a validation set to monitor model performance DURING training.
#
# IMPORTANT — Time Series Rule:
#   We DO NOT shuffle data or pick random samples.
#   Validation must also contain LATER time periods than training.
#
# So we do:
#   - The FIRST (90%) of X_train → X_train_final
#   - The LAST  (10%) of X_train → X_val
#
# This preserves correct temporal ordering and prevents future leakage.
# -------------------------------------------------------------
val_ratio = 0.1
n_train = X_train.shape[0]              # total number of pre-test training samples
val_size = int(n_train * val_ratio)     # number of validation samples

# -------------------------------------------------------------
# Final training set:
#   All training samples except the last 'val_size'
# -------------------------------------------------------------
X_train_final = X_train[:-val_size]
y_train_final = y_train[:-val_size]

# -------------------------------------------------------------
# Validation set:
#   The LAST 'val_size' samples from the training data
#   (chronologically later than the training set)
# -------------------------------------------------------------
X_val = X_train[-val_size:]
y_val = y_train[-val_size:]

print("Train final:", X_train_final.shape, y_train_final.shape)
print("Val        :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)


Train final: (1704447, 60, 15) (1704447,)
Val        : (189382, 60, 15) (189382,)
Test       : (473458, 60, 15) (473458,)


In [12]:
# -------------------------------------------------------------
# Feature Scaling (StandardScaler)
# -------------------------------------------------------------
# We normalize each feature so that:
#   • mean ≈ 0
#   • standard deviation ≈ 1
#
# This greatly stabilizes LSTM training and ensures that
# features with large numeric ranges do NOT dominate the model.
#
# IMPORTANT (NO DATA LEAKAGE):
#   The scaler is fit ONLY on the TRAINING DATA.
#   Validation and test data are transformed using the SAME scaler.
#   This prevents exposing the model to future information.
#
# Since the data has shape:
#   (num_samples, seq_len, num_features)
#
# We:
#   1) Flatten the time dimension (seq_len) into samples
#   2) Fit the scaler over ALL time steps in the training set
#   3) Apply the scaler to train/val/test and reshape back
# -------------------------------------------------------------

from sklearn.preprocessing import StandardScaler

num_features = X_train_final.shape[2]   # number of features per time step

# Create StandardScaler (mean=0, std=1)
scaler = StandardScaler()

# -------------------------------------------------------------
# Fit scaler ONLY on the training data
# -------------------------------------------------------------
# reshape (N, SEQ_LEN, num_features) → (N * SEQ_LEN, num_features)
# So we treat every time step as an independent sample for scaling.
X_train_2d = X_train_final.reshape(-1, num_features)

# Compute mean and std for each feature using ONLY training data
scaler.fit(X_train_2d)

# -------------------------------------------------------------
# Helper function to apply scaler to any dataset
# -------------------------------------------------------------
def apply_scaler(X):
    # X has shape (n_samples, seq_len, num_features)
    n, seq_len, nf = X.shape

    # Flatten time dimension: (n * seq_len, num_features)
    X_2d = X.reshape(-1, nf)

    # Apply scaler: subtract training mean, divide by training std
    X_scaled_2d = scaler.transform(X_2d)

    # Reshape back to 3D for LSTM input
    return X_scaled_2d.reshape(n, seq_len, nf)

# -------------------------------------------------------------
# Apply scaler to all splits
# -------------------------------------------------------------
X_train_scaled = apply_scaler(X_train_final)
X_val_scaled   = apply_scaler(X_val)
X_test_scaled  = apply_scaler(X_test)

# Final sanity check
print("Scaled shapes:")
print("  train:", X_train_scaled.shape)
print("  val  :", X_val_scaled.shape)
print("  test :", X_test_scaled.shape)


Scaled shapes:
  train: (1704447, 60, 15)
  val  : (189382, 60, 15)
  test : (473458, 60, 15)


In [76]:
# -------------------------------------------------------------
# PyTorch Dataset Wrapper for Time-Series Sequences
# -------------------------------------------------------------
# We define a custom Dataset to feed the LSTM batches of:
#     X : (seq_len, num_features)
#     y : scalar label (0/1)
#
# This dataset simply stores the pre-scaled NumPy arrays and
# converts them to PyTorch tensors.
#
# NOTE:
#   • y is stored as FLOAT because BCEWithLogitsLoss expects
#     floating-point targets (0.0 or 1.0), not integers.
# -------------------------------------------------------------

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, seq_len, num_features)
        # y: (N,)
        #
        # Convert NumPy arrays to PyTorch tensors.
        # .float() ensures the dtype is compatible with the model and loss.
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()  # float for BCEWithLogitsLoss

    def __len__(self):
        # Number of samples in the dataset
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# -------------------------------------------------------------
# Create training, validation, and test Datasets
# -------------------------------------------------------------
batch_size = 16

train_ds = TimeSeriesDataset(X_train_scaled, y_train_final)
val_ds   = TimeSeriesDataset(X_val_scaled,   y_val)
test_ds  = TimeSeriesDataset(X_test_scaled,  y_test)

# -------------------------------------------------------------
# Wrap datasets in DataLoaders
# -------------------------------------------------------------
# DataLoader handles batching, shuffling (for training), and
# iterating through the dataset efficiently.
#
# IMPORTANT:
#   • Training loader uses shuffle=True
#       → Good for SGD; ensures batches contain different samples.
#
#   • Validation and test loaders use shuffle=False
#       → Order must be consistent for reproducible evaluation.
# -------------------------------------------------------------
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

# Show dataset sizes
len(train_ds), len(val_ds), len(test_ds)


(1704447, 189382, 473458)

In [77]:
# -------------------------------------------------------------
# Device selection (CPU / CUDA / MPS)
# -------------------------------------------------------------
# We automatically pick the "best" available device:
#   1) Apple Silicon GPU (MPS) if available
#   2) NVIDIA GPU (CUDA) if available
#   3) Fallback to CPU otherwise
#
# This way the same code runs efficiently on different machines.
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
else:
    device = torch.device("cpu")
    print("Using device: CPU")


# -------------------------------------------------------------
# Basic shape information (for sanity + model config)
# -------------------------------------------------------------
# X_train_scaled has shape: (num_train_samples, seq_len, num_features)
# We read seq_len and num_features directly from the data to avoid
# hard-coding them.
# -------------------------------------------------------------
seq_len = X_train_scaled.shape[1]
num_features = X_train_scaled.shape[2]
print(f"Sequence length: {seq_len}, num_features: {num_features}")

Using device: MPS (Apple Silicon GPU)
Sequence length: 60, num_features: 15


##################################################################################

In [8]:
from Project.models import LSTMClassifier

# -------------------------------------------------------------
# Instantiate model, loss, and optimizer
# -------------------------------------------------------------
model = LSTMClassifier(
    input_size=num_features, hidden_size=64, num_layers=2, dropout=0.1
)

# Move model to selected device (CPU / CUDA / MPS)
model = model.to(device)

# Binary classification loss:
#   • Expects raw logits from the model
#   • Targets should be floats 0.0 or 1.0 (we already set y to float)
criterion = nn.BCEWithLogitsLoss()

# Adam optimizer with a moderate learning rate

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

NameError: name 'num_features' is not defined

In [ ]:
# -------------------------------------------------------------
# Final sanity checks before training
# -------------------------------------------------------------
# We print:
#   • the device being used (CPU / CUDA / MPS)
#   • number of training samples in the Dataset
#   • number of batches produced by the DataLoader
#   • batch size used for iteration
#
# These checks help confirm:
#   - the dataset loaded correctly
#   - the dataloader behaves as expected
#   - the batch size is applied properly
#   - the hardware accelerator is being used
# -------------------------------------------------------------
print("device:", device)                        # e.g., mps / cuda / cpu
print("len(train_ds):", len(train_ds))          # number of training sequences
print("len(train_loader):", len(train_loader))  # number of batches = ceil(len(train_ds) / batch_size)
print("batch_size:", batch_size)                # user-defined batch size


from training import train_model

# -------------------------------------------------------------
# Run the full training loop
# -------------------------------------------------------------
# train_model(...) performs:
#   1) For each epoch:
#        - one full training epoch  (model updated)
#        - one full validation epoch (no updates)
#   2) Tracks:
#        - training loss & accuracy
#        - validation loss & accuracy
#        - TP/FP/TN/FN per epoch
#   3) Saves the model with the **lowest validation loss**
#   4) Returns a "history" dictionary with all recorded metrics
#
# NOTE:
#   Setting epochs=1 trains only one epoch — useful for debugging.
#   Increase to ~10–20 for real training.
# ---

EPOCHS = 10
history = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=EPOCHS,                      # number of full passes over the data
    lr=1e-3,                            # optimizer learning rate
    model_path="best_lstm_model.pt",    # where the best weights get saved
)


device: mps
len(train_ds): 1704447
len(train_loader): 106528
batch_size: 16


Epochs:   0%|          | 0/10 [00:00<?, ?it/s]


[DEBUG] Starting epoch 1/10
[DEBUG]  Running train_epoch...
[DEBUG]  loader has 106528 batches


Train:   0%|          | 0/106528 [00:00<?, ?it/s]

[DEBUG]  Finished train_epoch.
[DEBUG]  loader has 11837 batches


...

[DEBUG]  Finished validation.
Epoch: 09 | Time: 23m 16s
	Train Loss: 0.454 | Train Acc: 76.13%
	 Val. Loss: 0.445 |  Val. Acc: 76.68%
	Train Confusion: TP=653311 FP=208068 TN=644304 FN=198764
	 Val. Confusion: TP=74476 FP=22724 TN=70733 FN=21449

[DEBUG] Starting epoch 10/10
[DEBUG]  Running train_epoch...
[DEBUG]  loader has 106528 batches


Train:   0%|          | 0/106528 [00:00<?, ?it/s]

[DEBUG]  Finished train_epoch.
[DEBUG]  loader has 11837 batches


Eval:   0%|          | 0/11837 [00:00<?, ?it/s]

[DEBUG]  Finished validation.
Epoch: 10 | Time: 23m 13s
	Train Loss: 0.455 | Train Acc: 76.05%
	 Val. Loss: 0.445 |  Val. Acc: 76.65%
	Train Confusion: TP=653010 FP=209204 TN=643168 FN=199065
	 Val. Confusion: TP=73975 FP=22270 TN=71187 FN=21950

[DEBUG] Training completed.



In [82]:
# -------------------------------------------------------------
# Load the best-performing model checkpoint
# -------------------------------------------------------------
# During training, we saved the model weights (state_dict) every
# time the validation loss improved.  This file contains ONLY the
# trained parameters — not the model class itself.
#
# To load it correctly, we must:
#   1. Recreate the SAME model architecture (same layers/sizes)
#   2. Load the saved state_dict into that model
#   3. Move the model to the correct device (CPU / GPU / MPS)
#
# NOTE:
#   • If ANY model hyperparameter changes (num_features, hidden_size,
#     num_layers, dropout), loading will fail.
#   • This pattern is standard in PyTorch: checkpoint = weights only.
# -------------------------------------------------------------

best_model = LSTMClassifier(
    input_size=num_features, hidden_size=64, num_layers=2, dropout=0.1
)

# Load weights from file into the model
best_model.load_state_dict(torch.load("best_lstm_model.pt", map_location=device))

# Ensure the model runs on the same device as the evaluation tensors
best_model.to(device)

LSTMClassifier(
  (lstm): LSTM(15, 64, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [83]:
# -------------------------------------------------------------
# 2) Accuracy + confusion per t_to_end_min
# -------------------------------------------------------------
# Here we go beyond global metrics and analyze performance
# as a function of "minutes to end of 15-min window".
#
# Steps:
#   1) Find where t_to_end_min lives in the feature vector
#   2) Extract t_to_end_min for each TEST sample
#      (from the *unscaled* X_test, last time step in each sequence)
#   3) Run a forward pass over test_loader to collect:
#         - predicted labels
#         - true labels
#   4) Build a DataFrame and compute:
#         - TP/FP/TN/FN per t_to_end_min
#         - accuracy per t_to_end_min
# -------------------------------------------------------------
import pandas as pd

# 2.1) Find the feature index for t_to_end_min
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# 2.2) Extract t_to_end_min from the UN-SCALED test data
#      Each sequence has shape (seq_len, num_features).
#      We take the LAST timestep [-1] for each sample:
#         → this corresponds to the "current" minute the model is predicting for.
t_to_end_min_values = X_test[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# 2.3) Collect predictions and true labels from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc="Predicting (test)", leave=False):
        X_batch = X_batch.to(device)

        # Forward pass
        logits = best_model(X_batch)
        probs = torch.sigmoid(logits)

        # Hard predictions (0/1) on CPU numpy arrays
        preds = (probs >= 0.5).float().cpu().numpy()
        all_preds.extend(preds)

        # y_batch is already float tensor (0.0/1.0); move to CPU numpy
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds).astype(int).reshape(-1)
all_labels = np.array(all_labels).astype(int).reshape(-1)

assert all_preds.shape[0] == t_to_end_min_values.shape[0], \
    "Mismatch: number of test predictions != number of t_to_end_min entries"


# 2.4) Build a DataFrame for per-minute analysis
df_results = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "y_pred": all_preds,
})

# Add confusion components per sample
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# 2.5) Aggregate stats by t_to_end_min
stats = df_results.groupby("t_to_end_min").agg(
    count=("correct", "count"),
    tp=("tp", "sum"),
    fp=("fp", "sum"),
    tn=("tn", "sum"),
    fn=("fn", "sum"),
    accuracy=("correct", "mean"),
).reset_index()

stats["accuracy_pct"] = stats["accuracy"] * 100

# 2.6) Print per-minute results
print("\n" + "="*70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("="*70)
print(stats.to_string(index=False))
print("="*70)

# 2.7) Baseline vs model
# Baseline (always predicting 1) accuracy = fraction of positives in test labels
baseline_acc = all_labels.mean()
model_acc    = (all_preds == all_labels).mean()

print(f"\nBaseline Accuracy (always predict 1): {baseline_acc:.4f}")
print(f"Model Test Accuracy (from preds):      {model_acc:.4f}")

t_to_end_min is at feature index: 14
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]


Predicting (test):   0%|          | 0/29592 [00:00<?, ?it/s]


ACCURACY + CONFUSION METRICS PER t_to_end_min
 t_to_end_min  count    tp   fp    tn   fn  accuracy  accuracy_pct
          1.0  31564 15570   64 15759  171  0.992555     99.255481
          2.0  31564 14856  924 14899  885  0.942688     94.268787
          3.0  31564 14307 1491 14332 1434  0.907331     90.733114
          4.0  31564 13851 1933 13890 1890  0.878881     87.888100
          5.0  31564 13381 2419 13404 2360  0.848593     84.859333
          6.0  31564 12975 2809 13014 2766  0.823375     82.337473
          7.0  31564 12691 3202 12621 3050  0.801926     80.192625
          8.0  31564 12240 3624 12199 3501  0.774268     77.426815
          9.0  31564 11748 4048 11775 3993  0.745248     74.524775
         10.0  31564 11252 4536 11287 4489  0.714073     71.407299
         11.0  31564 10819 5034 10789 4922  0.684577     68.457737
         12.0  31564 10393 5560 10263 5348  0.654416     65.441642
         13.0  31563 10092 6088  9734 5649  0.628141     62.814054
         14.0  